In [1]:
%load_ext watermark
%watermark -a "Manuel Elias Orellana Lavayen" -d -v -iv

Author: Manuel Elias Orellana Lavayen

Date: 2026-10-05

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.17.1

debugpy: 1.8.22



# Importando Librerias

In [2]:
import polars as pl
import pandas as pd
from pathlib import Path
import os
ruta_base = Path.cwd()

ruta_base

WindowsPath('C:/Users/ManuelOrellana/Desktop/Proyectos Aplicaciones/Aplicacion Prediccion Productos/Creacion de Modelos/Analisis y entrenamiento de modelos')

In [3]:
def verificar_valores_nulos_lazy(dataframe: pl.LazyFrame):
    """
    Verifica valores nulos de un LazyFrame de Polars para todas sus columnas

    Args:
        dataframe: LazyFrame de Polars

    Returns:
        No devuelve ningun valor, solo imprime la tabla resultante
    """
    resultado = (
        dataframe
        .select([
            pl.col(columna).null_count().alias(columna)
            for columna in dataframe.collect_schema().names()
        ])
        .collect(engine="streaming")
    )

    for columna in resultado.columns:
        print(f"{columna:<20} : {resultado[columna].item()}")

df_train = pl.scan_parquet(ruta_base / "Datasets/Concatenados/Sucios/train.parquet")

verificar_valores_nulos_lazy(df_train)
print(df_train.collect().dtypes)
df_train.head(3).collect()

ID                   : 0
fecha                : 0
numero_tienda        : 0
numero_articulo      : 0
unidades_vendidas    : 0
en_promocion         : 238945211
ciudad               : 0
provincia_estado     : 0
tipo_tienda          : 0
cluster              : 0
transacciones        : 483800
familia_producto     : 0
clase_producto       : 0
perecedero           : 0
precio_petroleo_wti  : 0
es_feriado_nacional  : 0
es_evento            : 0
es_feriado_regional  : 0
es_feriado_local     : 0
[UInt32, Date, Int8, Int32, Float32, Boolean, Categorical, Categorical, Categorical, Int8, Float32, Categorical, Int16, Int8, Float32, Boolean, Boolean, Boolean, Boolean]


ID,fecha,numero_tienda,numero_articulo,unidades_vendidas,en_promocion,ciudad,provincia_estado,tipo_tienda,cluster,transacciones,familia_producto,clase_producto,perecedero,precio_petroleo_wti,es_feriado_nacional,es_evento,es_feriado_regional,es_feriado_local
u32,date,i8,i32,f32,bool,cat,cat,cat,i8,f32,cat,i16,i8,f32,bool,bool,bool,bool
0,2013-01-01,25,103665,7.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""BREAD/BAKERY""",2712,1,94.279999,true,false,false,false
1,2013-01-01,25,105574,1.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0,94.279999,true,false,false,false
2,2013-01-01,25,105575,2.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0,94.279999,true,false,false,false


# Dataset "Train"

# ETAPA 1


In [4]:
def verificar_valores_nulos(dataframe: pl.DataFrame):
    """
    Verifica valores nulos de un DataFrame de Polars para todas sus columnas

    Args:
        dataframe: DataFrame de Polars

    Returns:
        No devuelve ningun valor, solo imprime la tabla resultante
    """
    columnas = dataframe.columns
    for columna in columnas:
        print(f"{columna:<20} : {dataframe[columna].null_count()}")

diccionario_tipos_columnas = {
    "numero_tienda": pl.Int8,
    "transacciones": pl.Float32,
}

df_train = pl.read_parquet(ruta_base / "Datasets/Concatenados/Sucios/train.parquet", columns= ["fecha","numero_tienda","transacciones","es_feriado_nacional","es_evento","es_feriado_regional" ,"es_feriado_local"])

# Eliminando Duplicados
df_train = df_train.unique()

df_train.head(3)

fecha,numero_tienda,transacciones,es_feriado_nacional,es_evento,es_feriado_regional,es_feriado_local
date,i8,f32,bool,bool,bool,bool
2017-01-12,39,1119.0,false,false,false,false
2017-02-23,49,2693.0,false,false,false,false
2014-11-24,39,1312.0,false,false,false,false


In [5]:
verificar_valores_nulos(df_train)

fecha                : 0
numero_tienda        : 0
transacciones        : 118
es_feriado_nacional  : 0
es_evento            : 0
es_feriado_regional  : 0
es_feriado_local     : 0


### Rellenando Nulos de Transancciones

#### Creación de columnas "año", "mes", "dia", "dia_semana"

In [6]:
df_train = df_train.with_columns(pl.col("fecha").dt.year().alias("año")).cast({"año": pl.Int16})
df_train = df_train.with_columns(pl.col("fecha").dt.month().alias("mes")).cast({"mes": pl.Int8})
df_train = df_train.with_columns(pl.col("fecha").dt.day().alias("dia")).cast({"dia": pl.Int8})
df_train = df_train.with_columns(pl.col("fecha").dt.weekday().alias("dia_semana")).cast({"dia_semana": pl.Int8})

#### Visualizando en que tipo de épocas se dan los nulos en transacciones para optar por tecnicas de rellenado

In [7]:
df_train.filter((df_train["transacciones"].is_null()) & ((df_train["es_feriado_nacional"] == True) | (df_train["es_feriado_regional"] == True) | (df_train["es_feriado_local"] == True) | (df_train["es_evento"] == True)))["fecha"].value_counts()

fecha,count
date,u32
2016-01-01,1


In [8]:
df_train.filter((df_train["transacciones"].is_null()) & ((df_train["es_feriado_nacional"] == False) | (df_train["es_feriado_regional"] == False) | (df_train["es_feriado_local"] == False) | (df_train["es_evento"] == False)))["fecha"].value_counts()

fecha,count
date,u32
2016-09-27,2
2016-01-04,39
2016-01-01,1
2016-01-03,53
2016-01-02,17
2014-01-02,1
2014-03-24,1
2013-06-19,4


##### Rellenado Para Feriados/Eventos

In [9]:
# Creando máscara 2016-01-01
mascara_filas_2016_1_1 =  df_train["fecha"] == pl.lit("2016-01-01").str.to_date("%Y-%m-%d")

# Creando máscara años diferentes al 2016-01-01
mascara_filas_diferentes_1_1 =  (df_train["año"] != 2016) & (df_train["mes"] == 1) & (df_train["dia"] == 1)

# Sacando media con base en la máscara de años diferentes
media_filas_2016_1_1 = df_train.filter(mascara_filas_diferentes_1_1).group_by("numero_tienda").agg(pl.col("transacciones").mean().alias("transacciones"))

# Rellenando nulos con base en la máscara
df_train = df_train.with_columns(
                    pl.when(mascara_filas_2016_1_1)# Similar a un IF
                                                .then( # Pimer resultado
                                                    pl.col("transacciones").fill_null(
                                                                                    pl.col("numero_tienda").cast(pl.Float64)
                                                                                    .replace(
                                                                                        old=media_filas_2016_1_1["numero_tienda"], # old -> Busca
                                                                                        new=media_filas_2016_1_1["transacciones"]  # new -> Reemplaza
                                                                                    )
                                                    )
                                                )
                        .otherwise(pl.col("transacciones")) # Similar a un ELSE -> Segundo Resultado -> ES decir los que son False los deja igual
                        .alias("transacciones") #Como se crea una columna nueva
                    )

##### Rellenado para NO Feriados/Eventos

In [10]:
def rellenar_transacciones_dias(dataset: pl.DataFrame, fecha: str):
    """
    Rellena valores nulos para la clumna de transacciones de un Dataframe de polars

    Args:
        dataframe: DataFrame de Polars
        fecha : Fecha la cual se quiere rellenar valores nulos

    Returns:
        Devuelve el DataFrame con las filas de esa respectiva fecha llenas
    """
    fecha_objeto = pl.lit(fecha).str.to_date("%Y-%m-%d")

    # Máscara booleana
    mascara_booleana = dataset["fecha"] == fecha_objeto

    anio = fecha_objeto.dt.year()
    mes = fecha_objeto.dt.month()
    dia_semana = fecha_objeto.dt.weekday()

    # Media por grupos
    media = dataset.filter( (dataset["mes"] == mes) &
                            (dataset["año"] == anio) &
                            (dataset["dia_semana"] == dia_semana) &
                            (dataset["transacciones"].is_not_null()) &
                                (
                                    (dataset["es_feriado_nacional"] == False) |
                                    (dataset["es_feriado_regional"] == False) |
                                    (dataset["es_feriado_local"] == False) |
                                    (dataset["es_evento"] == False)
                                )
                            ).group_by("numero_tienda").agg(pl.col("transacciones").mean().alias("transacciones"))

    # Rellenando
    dataset= dataset.with_columns(
                    pl.when(mascara_booleana)# Similar a un IF
                                                .then( # Pimer resultado
                                                    pl.col("transacciones").fill_null(
                                                                                    pl.col("numero_tienda").cast(pl.Float64)
                                                                                    .replace(
                                                                                        old=media["numero_tienda"], # old -> Busca
                                                                                        new=media["transacciones"]  # new -> Reemplaza
                                                                                    )
                                                    )
                                                )
                        .otherwise(pl.col("transacciones")) # Similar a un ELSE -> Segundo Resultado -> ES decir los que son False los deja igual
                        .alias("transacciones") #Como se crea una columna nueva
                    )
    return  dataset

In [11]:
df_train= rellenar_transacciones_dias(dataset= df_train, fecha = "2013-06-19")
df_train= rellenar_transacciones_dias(dataset= df_train, fecha = "2014-01-02")
df_train= rellenar_transacciones_dias(dataset= df_train, fecha = "2014-03-24")
df_train= rellenar_transacciones_dias(dataset= df_train, fecha = "2016-01-02")
df_train= rellenar_transacciones_dias(dataset= df_train, fecha = "2016-01-03")
df_train= rellenar_transacciones_dias(dataset= df_train, fecha = "2016-01-04")
df_train= rellenar_transacciones_dias(dataset= df_train, fecha = "2016-09-27")

In [12]:
df_train.drop(["año", "mes", "dia", "dia_semana", "es_feriado_nacional", "es_evento", "es_feriado_regional", "es_feriado_local"]).write_parquet(ruta_base/ "Datasets/Concatenados/Limpios/estapa1.parquet")

# ETAPA 2

In [13]:
def verificar_valores_nulos(dataframe: pl.DataFrame):
    """
    Verifica valores nulos de un DataFrame de Polars para todas sus columnas

    Args:
        dataframe: DataFrame de Polars

    Returns:
        No devuelve ningun valor, solo imprime la tabla resultante
    """
    columnas = dataframe.columns
    for columna in columnas:
        print(f"{columna:<20} : {dataframe[columna].is_null().sum()}")

df_train = pl.read_parquet(ruta_base / "Datasets/Concatenados/Sucios/train.parquet", columns= ["fecha","numero_tienda", "numero_articulo", "en_promocion"])

# Ordenamos el dataframe
df_train = df_train.sort(["fecha","numero_tienda", "numero_articulo"])

df_train.head(3)

fecha,numero_tienda,numero_articulo,en_promocion
date,i8,i32,bool
2013-01-01,25,96995,null
2013-01-01,25,99197,null
2013-01-01,25,103501,null


In [14]:
verificar_valores_nulos(df_train)

fecha                : 0
numero_tienda        : 0
numero_articulo      : 0
en_promocion         : 238945211


### Rellenando Promociones

In [15]:
# Columnas temporales de valores anteriores y posteriores
df_train = df_train.with_columns(
    pl.col("en_promocion")
    .shift(1)
    .over(["numero_tienda", "numero_articulo"])
    .alias("valor_fila_anterior"))

df_train = df_train.with_columns(
    pl.col("en_promocion")
    .shift(-1)
    .over(["numero_tienda", "numero_articulo"])
    .alias("valor_fila_posterior"))

# Máscara booleana de valores nulos en promocion
mascara_nulos_promocion = df_train["en_promocion"].is_null()

# Rellenando según vecinos
df_train = df_train.with_columns(pl.when( mascara_nulos_promocion & (pl.col("valor_fila_anterior") == True) & (pl.col("valor_fila_posterior") == True)).then(True).otherwise(pl.col("en_promocion")).alias("en_promocion"))

df_train = df_train.with_columns(pl.when( mascara_nulos_promocion & (pl.col("valor_fila_anterior") == False) & (pl.col("valor_fila_posterior") == False)).then(False).otherwise(pl.col("en_promocion")).alias("en_promocion"))

df_train = df_train.with_columns(pl.when( mascara_nulos_promocion & (pl.col("valor_fila_anterior") == True) & (pl.col("valor_fila_posterior").is_null())).then(True).otherwise(pl.col("en_promocion")).alias("en_promocion"))

df_train = df_train.with_columns(pl.when( mascara_nulos_promocion & (pl.col("valor_fila_anterior") == False) & (pl.col("valor_fila_posterior").is_null())).then(False).otherwise(pl.col("en_promocion")).alias("en_promocion"))

df_train = df_train.with_columns(pl.when( mascara_nulos_promocion & (pl.col("valor_fila_anterior").is_null()) & (pl.col("valor_fila_posterior") == True)).then(True).otherwise(pl.col("en_promocion")).alias("en_promocion"))

df_train = df_train.with_columns(pl.when( mascara_nulos_promocion & (pl.col("valor_fila_anterior").is_null()) & (pl.col("valor_fila_posterior") == False)).then(False).otherwise(pl.col("en_promocion")).alias("en_promocion"))

df_train["en_promocion"].is_null().sum()

216619305

In [16]:
# Crear un resumen de la promoción para calcular si estuvo en promoción en diferentes locales en ese mismo día para un mismo artículo
# De esa manera crear la promoción inferida, que es una columna Booleana que se calculó con base en las promociones de otros locales
resumen_promocion = (
    df_train
    .group_by(["fecha", "numero_articulo"])
    .agg(
        pl.col("en_promocion")
        .eq(True)
        .sum()
        .alias("cantidad_true"),

        pl.col("en_promocion")
        .eq(False)
        .sum()
        .alias("cantidad_false"),
    )
    .with_columns(
        pl.when(pl.col("cantidad_true") > pl.col("cantidad_false"))
        .then(pl.lit(True))
        .when(pl.col("cantidad_false") > pl.col("cantidad_true"))
        .then(pl.lit(False))
        .when(
            (pl.col("cantidad_true") == pl.col("cantidad_false"))
            & (pl.col("cantidad_true") > 0)
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(None, dtype=pl.Boolean))
        .alias("promocion_inferida")
    )
    .select([
        "fecha",
        "numero_articulo",
        "promocion_inferida"
    ])
)

In [17]:
# 1. Unir el resumen para traer la columna "promocion_inferida"
df_train = df_train.join(
    resumen_promocion.select(["fecha", "numero_articulo", "promocion_inferida"]),
    on=["fecha", "numero_articulo"],
    how="left"
)

# 2. Rellenar los nulos y contar cuántos nulos quedaron al final
df_train = df_train.with_columns(
    pl.coalesce(["en_promocion", "promocion_inferida"]).alias("en_promocion")
)

# 3. Contar nulos restantes en la columna
nulos_restantes = df_train["en_promocion"].null_count()
print(f"Nulos restantes: {nulos_restantes}")

Nulos restantes: 144911206


In [18]:
df_train = df_train.with_columns(pl.col("en_promocion").fill_null(False))

In [19]:
verificar_valores_nulos(df_train)

fecha                : 0
numero_tienda        : 0
numero_articulo      : 0
en_promocion         : 0
valor_fila_anterior  : 239048023
valor_fila_posterior : 238957838
promocion_inferida   : 144911206


In [20]:
df_train.drop(["valor_fila_posterior", "valor_fila_anterior", "promocion_inferida"]).write_parquet(ruta_base/ "Datasets/Concatenados/Limpios/estapa2.parquet")

## Uniendo Ambas Etapas

In [21]:
df_train = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Sucios/train.parquet").cast(diccionario_tipos_columnas)
    .drop(["transacciones", "en_promocion"])
    .join(
         pl.scan_parquet(ruta_base/ "Datasets/Concatenados/Limpios/estapa1.parquet").cast({"numero_tienda": pl.Int8, "transacciones": pl.Float32})
         .join(
            pl.scan_parquet(ruta_base/ "Datasets/Concatenados/Limpios/estapa2.parquet").cast({"numero_tienda": pl.Int8, "numero_articulo": pl.Int32}),
            on= ["fecha", "numero_tienda"],
            how = "left"),
         on= ["fecha", "numero_tienda", "numero_articulo"],
         how = "left")
    )

df_train.sink_parquet(ruta_base / "Datasets/Concatenados/Limpios/train.parquet", engine= "streaming")

In [22]:
verificar_valores_nulos_lazy(df_train)

ID                   : 0
fecha                : 0
numero_tienda        : 0
numero_articulo      : 0
unidades_vendidas    : 0
ciudad               : 0
provincia_estado     : 0
tipo_tienda          : 0
cluster              : 0
familia_producto     : 0
clase_producto       : 0
perecedero           : 0
precio_petroleo_wti  : 0
es_feriado_nacional  : 0
es_evento            : 0
es_feriado_regional  : 0
es_feriado_local     : 0
transacciones        : 0
en_promocion         : 0


In [23]:
os.remove(ruta_base/ "Datasets/Concatenados/Limpios/estapa1.parquet")
os.remove(ruta_base/ "Datasets/Concatenados/Limpios/estapa2.parquet")

# Dataset "Test"

In [24]:
def verificar_valores_nulos(dataframe: pd.DataFrame):
    columnas = dataframe.columns
    for columna in columnas:
        print(f"{columna:<20} : {dataframe[columna].isnull().sum()}")

df_test = pd.read_parquet(path= ruta_base / "Datasets/Concatenados/Sucios/test.parquet")
df_test["fecha"] = pd.to_datetime(df_test["fecha"])
df_test = df_test.sort_values(["numero_tienda", "numero_articulo", "fecha"])
df_test.head()

,id,fecha,numero_tienda,numero_articulo,en_promocion,ciudad,provincia_estado,tipo_tienda,cluster,familia_producto,clase_producto,perecedero,precio_petroleo_wti,es_feriado_nacional,es_evento,es_feriado_regional,es_feriado_local
0,125497040,2017-08-16,1,96995,False,Quito,Pichincha,D,13,GROCERY I,1093,0,46.80,False,False,False,False
210654,125707694,2017-08-17,1,96995,False,Quito,Pichincha,D,13,GROCERY I,1093,0,47.07,False,False,False,False
421308,125918348,2017-08-18,1,96995,False,Quito,Pichincha,D,13,GROCERY I,1093,0,48.59,False,False,False,False
631962,126129002,2017-08-19,1,96995,False,Quito,Pichincha,D,13,GROCERY I,1093,0,48.59,False,False,False,False
842616,126339656,2017-08-20,1,96995,False,Quito,Pichincha,D,13,GROCERY I,1093,0,48.59,False,False,False,False


In [25]:
verificar_valores_nulos(df_test)

id                   : 0
fecha                : 0
numero_tienda        : 0
numero_articulo      : 0
en_promocion         : 0
ciudad               : 0
provincia_estado     : 0
tipo_tienda          : 0
cluster              : 0
familia_producto     : 0
clase_producto       : 0
perecedero           : 0
precio_petroleo_wti  : 0
es_feriado_nacional  : 0
es_evento            : 0
es_feriado_regional  : 0
es_feriado_local     : 0


In [26]:
df_test.to_parquet(ruta_base / "Datasets/Concatenados/Limpios/test.parquet")